In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 1. Prepare data

## 1.1. Load data

In [3]:
data = fetch_california_housing()

In [4]:
data.data

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]])

In [5]:
data.data.shape

(20640, 8)

In [6]:
data.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894])

In [7]:
data.target.shape

(20640,)

## 1.2. Prepare train - test

In [8]:
X = data.data
y = data.target

In [9]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
X_train.shape

(16512, 8)

In [11]:
X_test.shape

(4128, 8)

In [12]:
y_train.shape

(16512,)

In [13]:
y_test.shape

(4128,)

## 1.3. Preprocess data

In [14]:
# Apply MinMaxScaler to selected features
min_max_scaler = MinMaxScaler()
X_train[:, [0, 1, 2, 3, 4, 5]] = min_max_scaler.fit_transform(X_train[:, [0, 1, 2, 3, 4, 5]])
X_test[:, [0, 1, 2, 3, 4, 5]] = min_max_scaler.transform(X_test[:, [0, 1, 2, 3, 4, 5]])

In [15]:
# Apply StandardScaler to selected features
standard_scaler = StandardScaler()
X_train[:, [6, 7]] = standard_scaler.fit_transform(X_train[:, [6, 7]])
X_test[:, [6, 7]] = standard_scaler.transform(X_test[:, [6, 7]])

## 1.4. Convert data to torch.tensor

In [16]:
# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# 2. Build Linear Regression model

In [17]:
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

In [18]:
model = LinearRegressionModel(input_dim=X_train.shape[1])

In [19]:
model

LinearRegressionModel(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)

In [20]:
criterion = nn.MSELoss()

In [21]:
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 3. Train model

In [22]:
num_epochs = 300

In [23]:
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print the loss
    if epoch % 50 == 0 or epoch == num_epochs - 1:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.3f}')

/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([16512])) that is different to the input size (torch.Size([16512, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [0/300], Loss: 6.905
Epoch [50/300], Loss: 1.362
Epoch [100/300], Loss: 1.347
Epoch [150/300], Loss: 1.342
Epoch [200/300], Loss: 1.339
Epoch [250/300], Loss: 1.338
Epoch [299/300], Loss: 1.338


# 4. Evaluate model

In [24]:
# Evaluate the model on the test data
model.eval()
with torch.no_grad():
    y_pred_train = model(X_train)
    train_loss = criterion(y_pred_train, y_train)
    print(f'Training Loss: {train_loss.item():.4f}')

    y_pred_test = model(X_test)
    test_loss = criterion(y_pred_test, y_test)
    print(f'Test Loss: {test_loss.item():.4f}')

Training Loss: 1.3378
Test Loss: 1.3116


/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([4128])) that is different to the input size (torch.Size([4128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [25]:
model.train()

LinearRegressionModel(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)